## Bronze: Flow B reference data.

Two static input files for the EEG financing burden allocation:
  - bmf.steueraufkommen_bund_2024 : Bund share of taxes per Bundesland
  - destatis.bevoelkerung_2024    : Population per Bundesland (Stichtag 31.12.2024)

These are small reference tables that update once a year, so we read them
directly rather than via Auto Loader. Streaming overhead isn't justified
for 16-row CSVs.

In [0]:
# ----- Steueraufkommen (BMF) -----------------------------------------------

steuern_path = "/Volumes/eeg_dev/raw_files/landing/bmf/2024/steueraufkommen_bund_2024.csv"

steuern = (
    spark.read
    .option("header", "true")
    .option("delimiter", ";")
    .option("encoding", "UTF-8")
    .option("inferSchema", "true")
    .csv(steuern_path)
)

(
    steuern.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("eeg_dev.bronze.bmf_steueraufkommen_bund_2024")
)

print("Loaded bmf_steueraufkommen_bund_2024:")
steuern.show(20, truncate=False)


# ----- Bevölkerung (Destatis) ----------------------------------------------

bev_path = "/Volumes/eeg_dev/raw_files/landing/destatis/2024/bevoelkerung_2024.csv"

bev = (
    spark.read
    .option("header", "true")
    .option("delimiter", ";")           # change to "," if your file uses commas
    .option("encoding", "UTF-8")        # change to "windows-1252" if Destatis CSV is cp1252
    .option("inferSchema", "true")
    .csv(bev_path)
    .select("Bundesland", "einwohner_2024")   # drop _c2
)

(
    bev.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("eeg_dev.bronze.destatis_bevoelkerung_2024")
)

print("Loaded destatis_bevoelkerung_2024:")
bev.show(20, truncate=False)

In [0]:
%sql
SELECT * FROM eeg_dev.bronze.bmf_steueraufkommen_bund_2024 ORDER BY bund_anteil_mio_eur_2024 DESC;

SELECT * FROM eeg_dev.bronze.destatis_bevoelkerung_2024;